In [19]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

In [20]:
load_dotenv()

True

In [21]:
model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [22]:
# Create a state

class LLMState(TypedDict):

    question: str
    answer: str

In [23]:
def llm_qa(state: LLMState) -> LLMState:

    # extract the question from state
    question = state['question']

    # form a prompt
    prompt = f'Answer the following question {question}'

    # ask that question to the LLM
    answer = model.invoke(prompt).content

    # update the answer in the state
    state['answer'] = answer

    return state

In [24]:
# Create our graph

graph = StateGraph(LLMState)

# add nodes
graph.add_node('llm_qa', llm_qa)

# add edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

# compile
workflow = graph.compile()

In [25]:
# execute

initial_state = {'question': 'How far is moon from the earth?'}

final_state = workflow.invoke(initial_state)

print(final_state['answer'])

[{'type': 'text', 'text': 'The average distance from the Earth to the Moon is about **384,400 kilometers** (or about **238,855 miles**). \n\nBecause the Moon travels in an elliptical (oval-shaped) orbit rather than a perfect circle, this distance constantly changes:\n* **Perigee (Closest point):** About 363,300 km (225,600 miles)\n* **Apogee (Farthest point):** About 405,500 km (251,984 miles)', 'extras': {'signature': 'El0KWwERTTIPaEej5cA24vQxV9V5kSv0ZMCXkooX/bBsItcCX1OAGwucxD3B4kRXxXel6j1YuHxp1vqGkqYcsicPxau8gJcAP7cQDwQ5DZuAY502X7gyNRcGLwJBuqY='}}]
